In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [3]:
import ast
from collections import Counter
from collections import defaultdict
import kagglehub
import nltk
from nltk.stem import WordNetLemmatizer
import pandas as pd
import os
import re
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import normalize

In [26]:
path = kagglehub.dataset_download("shuyangli94/food-com-recipes-and-user-interactions")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\pnosz\.cache\kagglehub\datasets\shuyangli94\food-com-recipes-and-user-interactions\versions\2


In [27]:
recipes = pd.read_csv(os.path.join(path, 'RAW_recipes.csv'))
interactions = pd.read_csv(os.path.join(path, 'RAW_interactions.csv'))

print(f'Recipes read: {recipes.shape}')
print(f'Interactions read: {interactions.shape}')

print(f'Recipes columns: {recipes.columns.tolist()}')
print(f'Interactions columns: {interactions.columns.tolist()}')

Recipes read: (231637, 12)
Interactions read: (1132367, 5)
Recipes columns: ['name', 'id', 'minutes', 'contributor_id', 'submitted', 'tags', 'nutrition', 'n_steps', 'steps', 'description', 'ingredients', 'n_ingredients']
Interactions columns: ['user_id', 'recipe_id', 'date', 'rating', 'review']


In [28]:
recipes.sample(10)

,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
78593,easy warm lentil salad,391101,40,1210564,2009-09-21,"['60-minutes-or-less', 'time-to-make', 'course...","[120.5, 0.0, 14.0, 2.0, 15.0, 0.0, 7.0]",5,['cook lentils according to package directions...,this recipe was featured in this months family...,"['lentils', 'artichoke hearts', 'scallions', '...",6
126118,low carb mushroom swiss omelet,274130,10,511388,2007-12-27,"['15-minutes-or-less', 'time-to-make', 'course...","[966.8, 126.0, 24.0, 54.0, 95.0, 227.0, 4.0]",9,"['wash , dry , and thinly slice the musrhooms'...","this is healthy and yummy, what more can you a...","['eggs', 'sea salt', 'butter', 'fresh mushroom...",5
45105,chicken parmigiana jaffle,207923,10,68960,2007-01-28,"['ham', '15-minutes-or-less', 'time-to-make', ...","[449.2, 33.0, 10.0, 49.0, 62.0, 54.0, 10.0]",8,"['plug in the sandwich maker to heat up', 'in ...",in australia toasted sandwiches with sealed ed...,"['chicken breasts', 'tomato paste', 'bread', '...",6
135979,minestra cabbage beans,103825,65,92641,2004-11-11,"['lactose', 'time-to-make', 'course', 'main-in...","[442.6, 17.0, 36.0, 10.0, 49.0, 8.0, 21.0]",12,"['wash and cut cabbage head', 'i usually quart...",this recipe is one from childhood. my grandmot...,"['cabbage', 'garlic', 'olive oil', 'salt and p...",8
10065,asparagus cheese squares with crescent crust,171183,51,89831,2006-06-02,"['60-minutes-or-less', 'time-to-make', 'course...","[135.3, 10.0, 7.0, 7.0, 11.0, 20.0, 4.0]",15,"['set oven to 375f', 'prepare an ungreased a 1...",this makes a ton but don't worry you won't hav...,"['refrigerated crescent dinner rolls', 'butter...",10
16874,banana pineapple nut bread,88046,80,128447,2004-04-01,"['time-to-make', 'course', 'main-ingredient', ...","[291.7, 25.0, 80.0, 4.0, 6.0, 9.0, 11.0]",9,"['combine flour , sugar , soda , cinnamon , pe...",a wonderful sweet bread. i always eat this wit...,"['flour', 'sugar', 'baking soda', 'ground cinn...",11
47691,chili onions,467927,15,266635,2011-11-10,"['weeknight', '15-minutes-or-less', 'time-to-m...","[347.1, 44.0, 38.0, 2.0, 5.0, 11.0, 7.0]",5,"['in a small skillet over medium-high heat , h...",the recipe suggests using these onions on burg...,"['canola oil', 'onion', 'chili powder', 'groun...",5
24190,black bean and sweet potato enchiladas,457771,30,1922706,2011-06-02,"['30-minutes-or-less', 'time-to-make', 'main-i...","[339.6, 10.0, 19.0, 20.0, 36.0, 14.0, 18.0]",9,"['preheat oven to 375', 'in skillet set over m...",an easy and satisfying recipe. leftovers rehe...,"['6-inch corn tortillas', 'black beans', 'swee...",14
211655,the best potato sausage cabbage soup,357242,75,1098582,2009-02-22,"['time-to-make', 'course', 'main-ingredient', ...","[711.1, 67.0, 20.0, 44.0, 36.0, 57.0, 20.0]",11,"['brown the sausage in a large skillet', 'remo...",my mom has cooked this as long as i can rememb...,"['bob evans sausage', 'celery ribs', 'cabbage'...",10
228663,ww fish and chips,238847,75,39835,2007-07-05,"['time-to-make', 'course', 'main-ingredient', ...","[409.8, 4.0, 12.0, 27.0, 91.0, 2.0, 16.0]",20,['pour the buttermilk into a zip-lock bag or m...,recipe source: ww magazine (may 2007) if follo...,"['fat-free buttermilk', 'fish fillets', 'potat...",13


In [29]:
interactions.sample(10)

,user_id,recipe_id,date,rating,review
287774,287420,352862,2010-06-05,5,These were yummy! Easy to make and even easier...
497423,260689,333164,2010-11-08,5,Very tasty and quick to make
83032,961653,213659,2011-06-28,5,"I forgot to rate this 2 weeks ago, shame on me..."
52779,7108,56816,2003-11-30,5,Very good! I used Kahlua and followed directi...
1108910,55380,89575,2007-02-09,5,My children declared these to be some of the b...
101698,1345247,51221,2009-08-08,0,I added on a garlic clove and a bunch of sprin...
1053758,746803,455683,2011-06-10,5,"We all liked this one, I served it with pork c..."
384836,317770,66619,2006-11-25,3,These were good. Not really the same as sausa...
87786,464327,97496,2007-11-29,5,We had a little potluck party in one of my cla...
18827,570804,51997,2012-08-20,3,I can't believe this recipe gets such wonderfu...


In [30]:
interactions = interactions[interactions['rating'] >= 4].reset_index(drop=True)

interactions.sample(10)

,user_id,recipe_id,date,rating,review
773409,225372,130085,2005-09-02,5,Oh I definitely like the simplicity of this on...
405987,238674,66409,2006-12-11,4,"Easy, light, sweet scone recipe.\nWhat a great..."
597509,207616,99245,2008-07-01,5,I'm trying this recipe because Kim said it was...
586613,37305,31242,2002-06-22,5,These are absolutely delicious muffins! They ...
96882,2304054,257766,2012-06-23,5,"I made these, instead of scalloped potatoes, t..."
174079,104295,173710,2006-06-22,5,We loved this one. I was concerned that it wo...
510486,64934,15242,2003-02-27,5,"Made this tonight as a side dish, but since th..."
190793,284180,323469,2008-11-03,5,I served this to a group and everybody loved i...
98877,1040631,75056,2009-04-08,5,The whole family loved this cake. I made it a...
982901,176615,93075,2005-12-25,4,Beautiful and colorful! Only change I made was...


In [31]:
nltk.download('wordnet', quiet=True)
lemmatizer = WordNetLemmatizer()

def clean_ingredient(ingredient: str) -> str:
    ingredient = ingredient.lower().strip()
    ingredient = re.sub(r'\d+', '', ingredient)
    ingredient = re.sub(r'[^\w\s]', '', ingredient)
    return ' '.join(lemmatizer.lemmatize(word, pos='n') for word in ingredient.split())

def safe_eval_list(val):
    if isinstance(val, str):
        try:
            parsed = ast.literal_eval(val)
            if isinstance(parsed, list):
                return parsed
        except (ValueError, SyntaxError):
            return []
    return []

recipes = recipes[['id', 'name', 'minutes', 'description', 'ingredients', 'steps']]
recipes['ingredients'] = recipes['ingredients'].apply(safe_eval_list)
recipes['steps'] = recipes['steps'].apply(safe_eval_list)
recipes['description'] = recipes['description'].fillna('')

recipes['cleaned_ingredients'] = recipes['ingredients'].apply(
    lambda ings: list({cleaned for ingredient in ings
    if (cleaned := clean_ingredient(ingredient))})
)

recipes.sample(10)

,id,name,minutes,description,ingredients,steps,cleaned_ingredients
197457,138453,spinach and mushroom quesadillas,30,"this is a recipe, i don't know where it came f...","[spinach, brie cheese, butter, garlic cloves, ...",[prepare spinach according to package directio...,"[garlic clove, portabella mushroom cap, brie c..."
59489,159741,cousin carolines baked beans,270,another recipe from mom's grey cookbook posted...,"[dried navy beans, butter, brown sugar, catsup...",[rinse beans and boil for 1 hour or until done...,"[dried navy bean, brown sugar, salt, molasses,..."
150498,168366,outback rice,40,hey hey the ex prep guy kickin out a few more ...,"[bay leaves, chicken base, water, rice, butter...",[i will teach you short cuts that will save yo...,"[red bell pepper, chicken base, rice, bay leaf..."
4921,118778,amish bob andy pie,60,this is really very good...,"[granulated sugar, milk, clove, eggs, flour, b...","[on the three large eggs , you will need to se...","[milk, butter, egg, flour, clove, granulated s..."
67205,17330,crunchy fudge sandwich,20,like a candy bar!,"[butterscotch chips, crunchy peanut butter, cr...",[melt butterscotch chips and peanut butter ove...,"[crunchy peanut butter, powdered sugar, butter..."
228872,251257,x rated,5,from absolut vodka.,"[vodka, grapefruit juice, white creme de cacao]",[fill a cocktail glass to the rim with crushed...,"[white creme de cacao, grapefruit juice, vodka]"
228690,364567,ww hearty stuffed bell peppers,75,"this is a classic, old-fashioned stuffed bell ...","[white rice, bell peppers, extra lean ground b...",[cook the rice according to package directions...,"[vegetable juice, garlic powder, salt, garlic ..."
209974,427163,tex mex goulash,45,this recipe makes much more than i anticipated...,"[lean ground beef, elbow macaroni, taco season...","[start by boiling the elbow noodles, while elb...","[lean ground beef, white onion, blackeyed pea,..."
152218,243756,pancake spread,7,"when i was younger, my sisters and i used to t...","[pancake syrup, confectioners' sugar, chocolat...","[mix all ingrediants together until smooth, yo...","[confectioner sugar, chocolate syrup, milk, pe..."
179622,450834,salted brown butter rice krispie treats,20,a reinvented rice crispy treat. when melting ...,"[unsalted butter, marshmallows, coarse sea sal...","[butter , or coat with non-stick spray , an 8-...","[coarse sea salt, marshmallow, cereal, unsalte..."


In [32]:
recipes.info()

<class 'pandas.DataFrame'>
RangeIndex: 231637 entries, 0 to 231636
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   id                   231637 non-null  int64 
 1   name                 231636 non-null  str   
 2   minutes              231637 non-null  int64 
 3   description          231637 non-null  str   
 4   ingredients          231637 non-null  object
 5   steps                231637 non-null  object
 6   cleaned_ingredients  231637 non-null  object
dtypes: int64(2), object(3), str(2)
memory usage: 61.8+ MB


In [33]:
print(f'Recipes before: {len(recipes)}')
print(f'Interactions before: {len(interactions)}')

valid_recipe_ids = set(recipes['id'])
interactions = interactions[interactions['recipe_id']
                            .isin(valid_recipe_ids)].reset_index(drop=True)

recipe_counts = interactions['recipe_id'].value_counts()
popular_recipes = set(recipe_counts[recipe_counts >= 10].index)
interactions = interactions[interactions['recipe_id']
                            .isin(popular_recipes)].reset_index(drop=True)

user_counts = interactions['user_id'].value_counts()
active_users = set(user_counts[user_counts >= 2].index)
interactions = interactions[interactions['user_id']
                            .isin(active_users)].reset_index(drop=True)

recipes = recipes[recipes['id'].isin(popular_recipes)].reset_index(drop=True)

print(f'Recipes after: {len(recipes)}')
print(f'Interactions after: {len(interactions)}')

Recipes before: 231637
Interactions before: 1003724
Recipes after: 18746
Interactions after: 427849


In [34]:
freq = Counter()
for ings in recipes['cleaned_ingredients']:
  freq.update(ings)

print(f'Total ingredients: {len(freq)}')

min_count = 10
ingredients = { ing for ing, count in freq.items() if count >= min_count }

print(f'Ingredients after filtering: {len(ingredients)}')

recipes['cleaned_ingredients'] = recipes['cleaned_ingredients'].apply(
  lambda ings: [ing for ing in ings if ing in ingredients]
)

recipes = recipes[recipes['cleaned_ingredients'].map(len) >= 3].reset_index(drop=True)

for ing in sorted(ingredients):
  print(ing)

Total ingredients: 5443
Ingredients after filtering: 1382
a original sauce
accent seasoning
acorn squash
active dry yeast
adobo sauce
alfredo sauce
allpurpose flour
allspice
almond
almond extract
aluminum foil
amaretto
american cheese
anchovy fillet
angel food cake
angel hair pasta
apple
apple cider
apple cider vinegar
apple juice
apple pie filling
applesauce
apricot jam
apricot preserve
arborio rice
artichoke
artichoke heart
artificial sweetener
asiago cheese
asparagus
asparagus spear
assorted fresh vegetable
avocado
baby carrot
baby potato
baby red potato
baby spinach
baby spinach leaf
bacon
bacon bit
bacon dripping
baguette
bailey irish cream
baked bean
baking cocoa
baking mix
baking potato
baking powder
baking soda
balsamic vinegar
bamboo shoot
banana
barbecue sauce
barley
basil
basil leaf
basil pesto
basmati rice
bay leaf
bean sprout
beef
beef bouillon
beef bouillon cube
beef bouillon granule
beef brisket
beef broth
beef consomme
beef gravy
beef roast
beef stew meat
beef stock
bee

In [35]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [36]:
ingredients = sorted(ingredients)

embeddings = model.encode(ingredients)
embeddings = normalize(embeddings, norm='l2')

clustering = AgglomerativeClustering(
    n_clusters=None,
    metric='cosine',
    linkage='complete',
    distance_threshold=0.15
)
labels = clustering.fit_predict(embeddings)

clusters = defaultdict(list)
for idx, label in enumerate(labels):
    clusters[label].append(ingredients[idx])

ingredient_to_canonical = {}
canonicals = {}
for members in clusters.values():
    members.sort(key=lambda x: (len(x), x))
    canonical = members[0]
    canonicals[canonical] = members
    for m in members:
        ingredient_to_canonical[m] = canonical

recipes['cleaned_ingredients'] = recipes['cleaned_ingredients'].apply(
    lambda ings: list({ ingredient_to_canonical[ing] for ing in ings })
)

print(f'Ingredients count before clusterization: {len(ingredients)}')
print(f'Ingredients count after: {len(canonicals)}')

for canonical, ings in canonicals.items():
    print(f'{canonical}: {ings}')

Ingredients count before clusterization: 1382
Ingredients count after: 1106
a original sauce: ['a original sauce']
accent seasoning: ['accent seasoning']
acorn squash: ['acorn squash']
dry yeast: ['dry yeast', 'active dry yeast', 'dry active yeast']
adobo sauce: ['adobo sauce']
alfredo sauce: ['alfredo sauce']
allpurpose flour: ['allpurpose flour']
allspice: ['allspice']
almond: ['almond']
almond extract: ['almond extract']
aluminum foil: ['aluminum foil']
amaretto: ['amaretto']
american cheese: ['american cheese']
anchovy fillet: ['anchovy fillet']
angel food cake: ['angel food cake']
angel hair pasta: ['angel hair pasta']
apple: ['apple']
apple cider: ['apple cider']
cider vinegar: ['cider vinegar', 'apple cider vinegar']
apple juice: ['apple juice']
apple pie filling: ['apple pie filling']
applesauce: ['applesauce']
apricot jam: ['apricot jam']
apricot preserve: ['apricot preserve']
arborio rice: ['arborio rice']
artichoke: ['artichoke']
artichoke heart: ['artichoke heart']
artifici

In [ ]:
recipes['minutes'].describe()

count    1.812700e+04
mean     1.185626e+05
std      1.595023e+07
min      0.000000e+00
25%      2.000000e+01
50%      4.000000e+01
75%      6.500000e+01
max      2.147484e+09
Name: minutes, dtype: float64

In [43]:
print(f'Number of recipes before: {len(recipes)}')
recipes = recipes[(recipes['minutes'] > 0) & (recipes['minutes'] <= 4320)]
print(f'Number of recipes after: {len(recipes)}')

Number of recipes before: 18105
Number of recipes after: 18018


In [44]:
recipes['minutes'].describe()

count    18018.000000
mean        78.495671
std        167.035024
min          1.000000
25%         22.000000
50%         40.000000
75%         65.000000
max       3240.000000
Name: minutes, dtype: float64

In [42]:
zero_min_recipes = recipes[recipes['minutes'] == 0]
print(f'Number of 0-minute recipes: {len(zero_min_recipes)}')
zero_min_recipes.sample(10)

Number of 0-minute recipes: 87


,id,name,minutes,description,ingredients,steps,cleaned_ingredients
1120,3806,baked cranberry acorn squash,0,,"[acorn squash, apple, fresh cranberries, orang...","[cut squash in half lengthwise, remove seeds, ...","[brown sugar, cranberry, butter, acorn squash,..."
16156,4049,sweet potato pie vi,0,note: this is the best sweet potato pie recipe...,"[sweet potatoes, sugar, evaporated milk, eggs,...","[combine all ingredients in mixing bowl, mix w...","[nutmeg, salt, pie shell, sweet potato, evapor..."
1620,773,bbq ranchero chicken salad,0,,"[boneless skinless chicken breast halves, barb...",[brush both sides of chicken with barbecue sau...,"[chili powder, red bell pepper, tomato, cumin,..."
9396,6521,justin s tuna salad,0,,"[eggs, mayonnaise, dill relish, louisiana hot ...","[combine eggs and dill relish, add the rest of...","[tuna, louisiana hot sauce, egg, mayonnaise]"
16310,48743,tasty salisbury steak,0,good! good! good! i serve the gravy over mashe...,"[ground beef, dry breadcrumbs, salt, pepper, e...",[mix together first 5 ingredients and shape in...,"[sliced mushroom, dry breadcrumb, salt, beef b..."
1132,825,baked fish with lemon black pepper,0,,"[fish fillet, salt, turmeric, cumin powder, fr...",[place fish fillets into a medium sized bowl a...,"[vegetable oil, salt, fish fillet, black peppe..."
12659,4219,peanut butter bread bread machine,0,this is wonderful toasted!,"[water, peanut butter, whole wheat flour, glut...",[put all ingredients in pan according to manuf...,"[brown sugar, dry yeast, salt, bread flour, pe..."
8431,3993,herb baked salmon,0,,"[vegetable oil cooking spray, salmon fillet, d...","[preheat oven to 350 f, line a baking tray wit...","[dijon mustard, egg white, fresh oregano, salm..."
8519,2462,home fried potatoes,0,,"[onions, potatoes, salt & pepper, oil]","[peel and cube potatoes, cook in oil in covere...","[oil, potato, salt pepper, onion]"
12118,2901,orange roughy with veggies,0,,"[zucchini, onion, mushroom, lemon slices, oliv...","[saute zucchini , onion , mushrooms and lemon ...","[mushroom, olive oil, garlic powder, lemon sli..."


In [47]:
print("Zduplikowane ID:", recipes['id'].duplicated().sum())

Zduplikowane ID: 0
